# 02 — Tiền xử lý dữ liệu Materials Project
Kiểm tra CIF, occupancy, ô tinh thể; loại dữ liệu lỗi/trùng; chọn tối đa 1000 mẫu; chia train/validation/test.

Chia tập theo **công thức rút gọn** để các đa hình cùng công thức không rơi vào nhiều tập (tránh rò rỉ).

**Đầu ra:** `data/processed/selected.csv` (có cột `split`), `excluded.csv`, `preprocess_report.json`.

In [ ]:
# Cài thư viện (chỉ chạy trên Colab; runtime Colab bị reset sau mỗi phiên nên mỗi notebook tự cài thứ nó cần)
import sys
if "google.colab" in sys.modules:
    %pip install -q pyscf pymatgen

In [ ]:
import os, sys, json, time
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/ml_xc_colab"
else:
    ROOT = os.environ.get("ML_XC_ROOT", os.getcwd())
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)          # để import được thư mục utils/
DATA, RAW, PROC = f"{ROOT}/data", f"{ROOT}/data/raw", f"{ROOT}/data/processed"
print("ROOT =", ROOT)

In [ ]:
import numpy as np, pandas as pd
from pymatgen.core import Structure
from utils import structure_utils as su, dft_utils as dft

MAX_ATOMS = 8              # giới hạn ô đơn vị theo chi phí PySCF PBC (xem ước lượng thời gian ở bước 03)
N_MAX = 1000
FRACTIONS = (0.8, 0.1, 0.1)    # train / val / test
SEED = 0
EXCLUDE_MAGNETIC = True        # lần chạy đầu dùng nspin=1; nguyên tố từ tính cần kiểm tra spin riêng
RUN_TYPES = None               # ví dụ ["r2SCAN"] để chỉ giữ một mức lý thuyết; None = giữ tất cả
CHECK_BASIS = dft.DEFAULT_SETTINGS["basis"]
CHECK_PSEUDO = dft.DEFAULT_SETTINGS["pseudo"]
GAP_INSUL = dft.DEFAULT_SETTINGS["gap_insul_eV"]

## 1. Đọc CIF và kiểm tra cấu trúc

In [ ]:
man = pd.read_csv(f"{RAW}/mp_manifest.csv")
report = {"n_manifest": int(len(man))}
excluded = []
structs, flag_rows = {}, []
for r in man.itertuples():
    if not isinstance(r.cif_path, str):
        excluded.append({"material_id": r.material_id, "reason": "không có CIF"}); continue
    try:
        s = su.read_structure(f"{RAW}/{r.cif_path}")
    except Exception as e:
        excluded.append({"material_id": r.material_id, "reason": f"CIF lỗi: {type(e).__name__}"}); continue
    f = su.check_structure(s, max_atoms=MAX_ATOMS)
    f["material_id"] = r.material_id
    if not f["ok"]:
        why = ("không trật tự/occupancy" if not (f["ordered"] and f["occupancy_ok"])
               else "ô không hợp lệ" if not f["cell_ok"]
               else f"quá {MAX_ATOMS} nguyên tử" if f["n_atoms"] > MAX_ATOMS
               else "nguyên tử quá gần" if f["min_dist_A"] < 0.7
               else "thể tích/nguyên tử quá lớn")
        excluded.append({"material_id": r.material_id, "reason": why})
        continue
    structs[r.material_id] = s
    flag_rows.append(f)
report["after_structure_checks"] = len(structs)
print("Sau kiểm tra cấu trúc:", len(structs), "/", len(man))
pd.DataFrame(excluded)["reason"].value_counts() if excluded else print("Không loại mẫu nào ở bước này")

## 2. Loại mẫu không tính được bằng PySCF (thiếu basis/giả thế GTH) và nguyên tố từ tính

In [ ]:
all_el = {sp.symbol for s in structs.values() for sp in s.composition.elements}
usable = dft.available_elements(all_el, CHECK_BASIS, CHECK_PSEUDO)
print("Nguyên tố thiếu dữ liệu:", sorted(all_el - usable))
keep = {}
for mid, s in structs.items():
    els = {sp.symbol for sp in s.composition.elements}
    if not els <= usable:
        excluded.append({"material_id": mid, "reason": "thiếu basis/giả thế: " + ",".join(sorted(els - usable))})
    elif EXCLUDE_MAGNETIC and els & dft.MAGNETIC:
        excluded.append({"material_id": mid, "reason": "nguyên tố từ tính (cần spin riêng)"})
    else:
        keep[mid] = s
structs = keep
report["after_dft_feasibility"] = len(structs)
print("Sau lọc khả thi DFT:", len(structs))

## 3. Loại cấu trúc trùng và lọc run_type

In [ ]:
df = man[man["material_id"].isin(structs)].copy()
if RUN_TYPES is not None:
    drop_rt = df[~df["run_type"].isin(RUN_TYPES)]["material_id"]
    excluded += [{"material_id": m, "reason": "run_type ngoài danh sách"} for m in drop_rt]
    df = df[df["run_type"].isin(RUN_TYPES)]
ids, dup = su.dedupe_structures(df, structs)
excluded += [{"material_id": m, "reason": f"trùng với {k}"} for m, k in dup.items()]
df = df[df["material_id"].isin(ids)].copy()
report["after_dedup"] = int(len(df))
print("Sau loại trùng:", len(df), "| số cặp trùng:", len(dup))
print("\nrun_type còn lại:", df["run_type"].value_counts().to_dict(),
      "\n(band gap giữa các mức lý thuyết khác nhau không so sánh trực tiếp được)")

## 4. Chọn tối đa 1000 mẫu và chia train/validation/test

In [ ]:
df["reduced_formula"] = [structs[m].composition.reduced_formula for m in df["material_id"]]
df["n_atoms"] = [len(structs[m]) for m in df["material_id"]]
df["n_elements"] = [len(structs[m].composition.elements) for m in df["material_id"]]
df["n_elements_capped"] = df["n_elements"].clip(upper=3)
df["band_gap_eV"] = df["band_gap_eV"].fillna(0.0)
df["is_metal"] = (df["band_gap_eV"] < GAP_INSUL).astype(int)

sel = su.stratified_select(df, N_MAX, seed=SEED)
sel["split"] = su.group_split(sel, "reduced_formula", FRACTIONS, seed=SEED).values
cols = ["material_id", "formula_pretty", "reduced_formula", "n_atoms", "n_elements", "band_gap_eV", "is_metal",
        "energy_above_hull_eV_per_atom", "run_type", "task_id", "mp_database_version", "split"]
sel = sel[cols]

# kiểm tra không rò rỉ: mỗi công thức rút gọn chỉ nằm trong một split
assert sel.groupby("reduced_formula")["split"].nunique().max() == 1
report["n_selected"] = int(len(sel))
report["split_counts"] = sel["split"].value_counts().to_dict()
report["metal_fraction"] = float(sel["is_metal"].mean())
print(report)

In [ ]:
sel.to_csv(f"{PROC}/selected.csv", index=False)
pd.DataFrame(excluded).to_csv(f"{PROC}/excluded.csv", index=False)
report["settings"] = dict(MAX_ATOMS=MAX_ATOMS, N_MAX=N_MAX, FRACTIONS=FRACTIONS, SEED=SEED,
                          EXCLUDE_MAGNETIC=EXCLUDE_MAGNETIC, RUN_TYPES=RUN_TYPES,
                          basis_checked=CHECK_BASIS, pseudo_checked=CHECK_PSEUDO)
json.dump(report, open(f"{PROC}/preprocess_report.json", "w"), indent=2, ensure_ascii=False)
print("Đã lưu selected.csv, excluded.csv, preprocess_report.json")
sel.groupby("split")[["n_atoms", "band_gap_eV"]].describe().round(2)